### Imports

In [4]:
from transformers import AutoTokenizer, BertForMultipleChoice
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, load_from_disk


2025-12-01 15:58:54.118481: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-01 15:58:54.717667: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-01 15:58:57.803716: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Load data

In [9]:
dataset = load_dataset('csv', data_files={'train':"../data/clean/twitter_train_balanced.csv",
                                          'validation':"../data/clean/twitter_validation_clean.csv",
                                          'test':"../data/clean/twitter_test_clean.csv",})
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'Tweet Id', 'Entity', 'Sentiment', 'Tweet Content'],
        num_rows: 160028
    })
    validation: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'Tweet Id', 'Entity', 'Sentiment', 'Tweet Content'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'Tweet Id', 'Entity', 'Sentiment', 'Tweet Content'],
        num_rows: 1000
    })
})

In [10]:
dataset['train'][0:4]

{'Unnamed: 0.1': [0, 1, 2, 3],
 'Unnamed: 0': [53134, 23368, 28323, 20579],
 'Tweet Id': [2015, 4442, 518, 12756],
 'Entity': ['CallOfDuty', 'Google', 'ApexLegends', 'WorldOfCraft'],
 'Sentiment': ['Negative', 'Neutral', 'Negative', 'Neutral'],
 'Tweet Content': ['... This is chilling',
  'one',
  'literally toxic bro, came out of the game when I was clear to him. Oh and I hit the same person three times to talk about lmao disappointment.',
  'Take a look at this article I just got! [Uncanny Combat Gloves of the Incomparable Fighter]']}

### Encode sentiment and Entities
Convert sentiments to encoded values (0-3) and concatenate entitites to start of tweet content

In [ ]:
sentiments = ["Irrelevant","Positive","Neutral","Negative"]

# remove extraneous columns from loading and saving dataframes
dataset = dataset.remove_columns(['Unnamed: 0.1', 'Unnamed: 0', 'Tweet Id'])

print("Encoding sentiment")

for subset in ["train","validation","test"]:
    labels = []
    for d in tqdm(dataset[subset]["Sentiment"], desc=subset):
        for s in range(len(sentiments)):
            if sentiments[s] == d:
                labels.append(s)
                break
    dataset[subset] = dataset[subset].add_column("labels",labels)

print("Combining entities and text")
for subset in ["train","validation","test"]:
    combined = []
    for d in tqdm(range(len(dataset[subset])), desc=subset):
        combined.append(dataset[subset]["Entity"][d]+". "+dataset[subset]["Tweet Content"][d])
    dataset[subset] = dataset[subset].add_column("inputs",combined)

dataset = dataset.remove_columns(["Entity","Tweet Content"])

#ds_create()

dataset

### Save dataset to disk

In [ ]:
dataset = dataset.remove_columns(["Sentiment"])
dataset.save_to_disk("../data/bert-ds-labeled")
dataset